# Laboratorio 1: Il RAG in trenta righe

---

# Lab 1: RAG in thirty lines

**Come si usa.** Esegui le celle dall'alto in basso (Maiusc+Invio). Serve un account Google gratuito
per usare Colab. Il testo completo del laboratorio e' sul sito: https://corso-rag.giuseppebosi.com/moduli/01-perche-serve-una-memoria/LAB/

---

**How to use it.** Run the cells from top to bottom (Shift+Enter). Colab needs a free Google account.
The full text of the lab is on the site: https://corso-rag.giuseppebosi.com/en/moduli/01-perche-serve-una-memoria/LAB/

In [ ]:
# Prepara l'ambiente / Set up the environment (about 1 minute)
import os, sys, subprocess
if sys.version_info[:2] not in [(3, 11), (3, 12)]:
    print("ATTENZIONE: questo corso e' provato con Python 3.11 o 3.12; hai "
          f"{sys.version_info.major}.{sys.version_info.minor}. "
          "WARNING: the course is tested on Python 3.11 or 3.12.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(BASE, "rag-chatbot-course")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/JoeBosi/rag-chatbot-course.git", REPO_DIR], check=True)
os.chdir(os.path.join(REPO_DIR, "corso"))
!pip install -q openai python-dotenv httpx PyYAML pypdf
print("Cartella di lavoro / working folder:", os.getcwd())

## La chiave gratuita / The free key

Questo laboratorio chiama un modello: serve una chiave **Groq**, gratuita e senza carta di credito
([CHIAVI.md](https://corso-rag.giuseppebosi.com/CHIAVI/) spiega come ottenerla in due minuti).
Poi in Colab: icona della **chiave** nella barra a sinistra, **Aggiungi nuovo segreto**, nome
`GROQ_API_KEY`, valore la tua chiave, e attiva **Accesso da notebook**. Se non la imposti, la cella
te la chiede al momento. La chiave non finisce mai nel notebook.

---

This lab calls a model, so you need a free **Groq** key, no credit card
([CHIAVI.md](https://corso-rag.giuseppebosi.com/en/CHIAVI/) shows how to get one in two minutes).
Then in Colab: the **key** icon in the left bar, **Add new secret**, name `GROQ_API_KEY`, value your
key, and switch on **Notebook access**. If you skip this, the cell asks for the key when you run it.
The key is never saved in the notebook.

In [ ]:
# La chiave Groq / The Groq key. Mai scritta nel notebook / never written in the notebook.
import os
from getpass import getpass

def leggi_chiave(nome="GROQ_API_KEY"):
    if os.environ.get(nome):
        return os.environ[nome]
    try:
        from google.colab import userdata          # Colab: icona della chiave, a sinistra
        return userdata.get(nome)
    except Exception:
        pass
    try:
        return getpass(f"Incolla {nome} / paste {nome}: ")
    except Exception:
        return ""

chiave = leggi_chiave()
if chiave:
    os.environ["GROQ_API_KEY"] = chiave
    os.environ["LLM_PROVIDER"] = "groq"
    print("Chiave caricata (non viene mostrata) / key loaded (not displayed).")
else:
    print("Nessuna chiave. I passaggi che chiamano il modello si fermeranno con "
          "il messaggio 'Manca GROQ_API_KEY'. Vedi la cella sopra.\n"
          "No key. Steps that call the model will stop with 'Manca GROQ_API_KEY'. See the cell above.")
![ -f .env ] || cp .env.example .env

Le celle con un'etichetta **[serve la chiave]** chiamano il modello. Senza chiave si fermano con
`Manca GROQ_API_KEY in .env`: non e' un guasto, inserisci la chiave come spiegato sopra e rilancia.
Le altre celle funzionano anche senza.

---

Cells labelled **[needs the key]** call the model. Without a key they stop with
`Manca GROQ_API_KEY in .env` (the message is Italian): this is not a fault, add the key as explained
above and run the cell again. The other cells work without it.

## Passo 0: la cartella del tuo sistema

---

## Step 0: the folder of your system

## Starter o soluzione? / Starter or solution?

Nel laboratorio scrivi tu alcune funzioni (i TODO). Qui sotto scegli:
`USA_SOLUZIONE = True` copia la soluzione, cosi' tutte le celle girano subito;
`USA_SOLUZIONE = False` copia lo **starter** con i TODO: apri il file nel pannello **File** a sinistra
(cartella `rag-chatbot-course/corso/memoria`), scrivi la funzione, salva e rilancia le celle.

---

In the lab you write some functions yourself (the TODOs). Choose below:
`USA_SOLUZIONE = True` copies the solution so every cell runs straight away;
`USA_SOLUZIONE = False` copies the **starter** with the TODOs: open the file in the **Files** panel on
the left (folder `rag-chatbot-course/corso/memoria`), write the function, save, and run the cells again.

In [ ]:
USA_SOLUZIONE = True   # False = parti dallo starter e scrivi tu i TODO
MOD = "moduli/01-perche-serve-una-memoria"
ORIGINE = "soluzione" if USA_SOLUZIONE else "starter"
!mkdir -p memoria
!cp $MOD/$ORIGINE/llm.py memoria/
!cp $MOD/$ORIGINE/memoria_minima.py memoria/

Controlla il collegamento col modello **[serve la chiave]**. Deve stampare `pronto`.

---

Check the link to the model **[needs the key]**. It must print `pronto`.

In [ ]:
!python memoria/llm.py

## Passo 1: misura il problema

Sul corpus del corso sono circa 4.400 token.

---

## Step 1: measure the problem

On the course corpus it is about 4,400 tokens.

In [ ]:
!python moduli/01-perche-serve-una-memoria/starter/conta_token.py progetto-finale/corpus/manuali/*.md progetto-finale/corpus/note/*.md

## Passo 2: `cerca()`

Senza chiamare il modello. Risultato atteso: `manuali/greencut-rm46.md` e `note/2026-07-21-rasaerba-non-parte.md`, entrambi con 2 punti.

---

## Step 2: `cerca()`

Without calling the model. Expected: `manuali/greencut-rm46.md` and `note/2026-07-21-rasaerba-non-parte.md`, both with 2 points.

In [ ]:
import sys; sys.path.insert(0, "memoria")
from memoria_minima import carica_documenti, cerca
for d in cerca("Che candela monta il rasaerba?", carica_documenti()):
    print(d["punteggio"], d["nome"])

## Passo 4: provalo **[serve la chiave]**

Prima l'elenco dei documenti usati, poi una risposta che cita 0,8 bar e il ripristino a 1,2 bar.

---

## Step 4: try it **[needs the key]**

First the list of documents used, then an answer that cites 0.8 bar and the reset to 1.2 bar.

In [ ]:
!python memoria/memoria_minima.py "Cosa significa l'errore E4 sulla caldaia Termovex?"

## Passo 5: rompilo

Guarda l'**elenco dei documenti usati**, non la risposta. La prima domanda mette al primo posto la nota sui freni della bici, la seconda una nota sulla lavatrice, la terza tre documenti a pari punteggio.

---

## Step 5: break it

Look at the **list of documents used**, not the answer. The first question ranks the bike-brake note first, the second a washing-machine note, the third three documents with equal scores.

In [ ]:
!python memoria/memoria_minima.py "Quanto devo stringere la vite che tiene la lama?"
!python memoria/memoria_minima.py "Perche' il trapano avvitatore si scarica subito?"
!python memoria/memoria_minima.py "Il tagliaerba non si accende, cosa devo controllare?"

## Passo 6: verifica **[serve la chiave]**

Deve stampare `OK` su tutte le righe.

---

## Step 6: check **[needs the key]**

It must print `OK` on every line.

In [ ]:
!python moduli/01-perche-serve-una-memoria/verifica.py